# ANN and CNN Tryout
Case 1: tabular binary classification (ANN). Case 2: image binary classification (CNN).

Plots go to `result/case<x>/`, best models to `result/case<x>/model/`.

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from keras import layers
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "impl" else Path.cwd()
DATA, RESULT = ROOT / "data", ROOT / "result"
tf.get_logger().setLevel("ERROR")
keras.utils.set_random_seed(42)

In [ ]:
def save(fig, case, name):
    out = RESULT / case
    out.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out / f"{name}.png", dpi=120)
    plt.show()


def callbacks(case, name, patience):
    out = RESULT / case / "model"
    out.mkdir(parents=True, exist_ok=True)
    return [
        keras.callbacks.ModelCheckpoint(str(out / name), save_best_only=True),
        keras.callbacks.EarlyStopping(patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=max(3, patience // 3)),
    ]


def plot_history(h, case):
    print(f"Trained {len(h.history['loss'])} epochs")
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    for ax, k in zip(axes, ("loss", "accuracy")):
        ax.plot(h.history[k], label="train")
        ax.plot(h.history[f"val_{k}"], label="val")
        ax.set(title=k.title(), xlabel="Epoch")
        ax.legend()
    save(fig, case, "history")


def evaluate(model, x, y, labels, case):
    p = model.predict(x, verbose=0).ravel()
    print(classification_report(y, p > 0.5, target_names=labels, digits=3))
    print(f"ROC-AUC: {roc_auc_score(y, p):.3f}")
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay.from_predictions(
        y, p > 0.5, display_labels=labels, cmap="Blues", colorbar=False, ax=ax
    )
    save(fig, case, "confusion_matrix")

# Case 1 (ANN)

In [ ]:
df = pd.read_csv(DATA / "case1" / "DATA1.csv", index_col=0)
print(
    df.shape,
    "| duplicates:",
    df.duplicated().sum(),
    "| missing:",
    df.isna().sum().sum(),
)
df.describe().T.round(2)

In [ ]:
num = df.drop(columns="Class_att")
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
df["Class_att"].value_counts().plot.bar(
    ax=axes[0], color=["tab:red", "tab:green"], rot=0, title="Class count"
)

axes[0].bar_label(axes[0].containers[0])
sns.boxplot(num, orient="h", ax=axes[1]).set_title("Feature distribution")
num.corrwith(df["Class_att"].eq("Abnormal")).plot.barh(
    ax=axes[2], title="Correlation with Abnormal"
)

save(fig, "case1", "eda")

Col7–Col12 have |r| ≤ 0.1 with the target and near-zero mutual information. Dropping them raised 5-fold CV accuracy from ~0.82 to ~0.86. Scaling is done inside the model (`Normalization` layer), so the saved model takes raw features.

In [ ]:
FEATURES = [f"Col{i}" for i in range(1, 7)]
X, y = (
    df[FEATURES].to_numpy("float32"),
    df["Class_att"].eq("Abnormal").to_numpy("int32"),
)

x_tr, x_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

x_tr, x_val, y_tr, y_val = train_test_split(
    x_tr, y_tr, test_size=0.15 / 0.85, stratify=y_tr, random_state=42
)

print(f"train {len(x_tr)} | val {len(x_val)} | test {len(x_te)}")

In [ ]:
norm = layers.Normalization()
norm.adapt(x_tr)

ann = keras.Sequential(
    [
        keras.Input((len(FEATURES),)),
        norm,
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ]
)
ann.compile(keras.optimizers.Adam(1e-3), "binary_crossentropy", metrics=["accuracy"])

h1 = ann.fit(
    x_tr,
    y_tr,
    validation_data=(x_val, y_val),
    epochs=300,
    batch_size=16,
    verbose=0,
    callbacks=callbacks("case1", "ann.keras", patience=20),
)

plot_history(h1, "case1")

In [ ]:
evaluate(ann, x_te, y_te, ["Normal", "Abnormal"], "case1")

### Inference example

In [ ]:
model1 = keras.models.load_model(RESULT / "case1" / "model" / "ann.keras")
sample = pd.DataFrame(x_te[:5], columns=FEATURES)
prob = model1.predict(sample.to_numpy("float32"), verbose=0).ravel()
sample.assign(
    actual=np.where(y_te[:5], "Abnormal", "Normal"),
    p_abnormal=prob.round(3),
    predicted=np.where(prob > 0.5, "Abnormal", "Normal"),
)

# Case 2 (CNN)
The `train` folder is split 85/15 into train/val. The `valid` folder (200 images) is the test set, and `test` (6 images) is used for the inference example.

Note: many `valid`/`test` images are near-duplicates of `train` images, so scores here are optimistic.

In [ ]:
IMG, BS, C2 = (64, 64), 32, DATA / "case2" / "DATA2"
kw = {
    "label_mode": "binary",
    "image_size": IMG,
    "color_mode": "grayscale",
    "batch_size": None,
    "verbose": False,
}

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    str(C2 / "train"), validation_split=0.15, subset="both", seed=42, **kw
)

test_ds = keras.utils.image_dataset_from_directory(
    str(C2 / "valid"), shuffle=False, **kw
)

CLASSES = train_ds.class_names
prep = lambda ds, shuffle=False: (
    (ds.cache().shuffle(4096, seed=42) if shuffle else ds.cache())
    .batch(BS)
    .prefetch(tf.data.AUTOTUNE)
)

train_ds, val_ds, test_ds = prep(train_ds, True), prep(val_ds), prep(test_ds)

x, yb = next(iter(train_ds))
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, img, lbl in zip(axes.flat, x, yb):
    ax.imshow(img[..., 0], cmap="gray")
    ax.set_title(CLASSES[int(lbl[0])])
    ax.axis("off")

save(fig, "case2", "samples")

In [ ]:
conv = lambda f: [
    layers.Conv2D(f, 3, padding="same", use_bias=False),
    layers.BatchNormalization(momentum=0.9),
    layers.ReLU(),
    layers.MaxPooling2D(),
]

cnn = keras.Sequential(
    [
        keras.Input((*IMG, 1)),
        layers.Rescaling(1 / 255),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        *[l for f in (32, 64, 128, 256) for l in conv(f)],
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ]
)
cnn.compile(keras.optimizers.Adam(1e-3), "binary_crossentropy", metrics=["accuracy"])

h2 = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,
    verbose=0,
    callbacks=callbacks("case2", "cnn.keras", patience=15),
)

plot_history(h2, "case2")

In [ ]:
y_test = np.concatenate([y for _, y in test_ds]).ravel()
evaluate(cnn, test_ds, y_test, CLASSES, "case2")

### Inference example

In [ ]:
model2 = keras.models.load_model(RESULT / "case2" / "model" / "cnn.keras")
paths = sorted(p for p in (C2 / "test").rglob("*") if p.is_file())
imgs = np.stack(
    [
        keras.utils.img_to_array(
            keras.utils.load_img(
                p, color_mode="grayscale", target_size=IMG, interpolation="bilinear"
            )
        )
        for p in paths
    ]
)

prob = model2.predict(imgs, verbose=0).ravel()
fig, axes = plt.subplots(1, len(paths), figsize=(2.2 * len(paths), 2.8))
for ax, p, img, pr in zip(axes, paths, imgs, prob):
    ax.imshow(img[..., 0], cmap="gray")
    ax.axis("off")
    ax.set_title(
        f"true: {p.parent.name}\npred: {CLASSES[int(pr > 0.5)]} ({pr:.2f})", fontsize=9
    )

save(fig, "case2", "inference")